# Run one module on a clip

Each department module takes its own video. This notebook builds the event the desk expects from frame counts, which is the same function the model service uses after it has counted people and vehicles.

The detector is YOLOv8n on CPU. It counts people and vehicles. It has no weapon class, no crash class, and no plate reader, so those modules leave the decision with the officer. Movement is a jump in the people count, marked for review.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from cityvigilance.analyze import build_event

## Crowd clip

Two frames from a crowd video. The event keeps the peak, 9 people, not the average.

In [2]:
event = build_event("crowd", [{"person": 4}, {"person": 9, "car": 1}], "2026-11-02T12:00:00Z")
print(event["cameraId"], event["source"], event["finding"])
print(event["note"])

cam-crowd local {'type': 'crowd', 'count': 9}
Peak of 9 people the detector could separate, across 2 frames. Overlapping people in a dense crowd are undercounted.


## Traffic clip

Vehicles in the busiest frame set the proof-of-concept green time: twice the count, kept between 10 and 60 seconds. Metres are not measured.

In [3]:
event = build_event("traffic", [{"car": 6, "motorcycle": 2}, {"car": 3}], "2026-11-02T12:00:00Z")
print(event["finding"]["arms"][0])

{'name': 'In view', 'queueMetres': 0, 'vehicles': 8, 'currentGreenSeconds': 20, 'greenSeconds': 16}


## Weapon and accident clips

A video for either module is stored with the event, and the finding stays undecided until that model exists.

In [4]:
weapon = build_event("weapon", [{"person": 9}], "2026-11-02T12:00:00Z")
accident = build_event("accident", [{"car": 2}], "2026-11-02T12:00:00Z")
print(weapon["finding"], weapon["note"])
print(accident["finding"]["crash"], accident["note"])

{'type': 'weapon', 'kind': 'none'} No weapon model is loaded. The officer watches this clip. A later weapon model must return firearm, blade, or none.
None No accident model is loaded. The officer watches this clip. This desk does not call 112.


## Movement clip

Sudden means the busiest frame has at least 8 people and at least twice the quietest frame. A steady crowd stays steady. The level is a review, not an order.

In [5]:
sudden = build_event("movement", [{"person": 3}, {"person": 12}], "2026-11-02T12:00:00Z")
steady = build_event("movement", [{"person": 8}, {"person": 10}], "2026-11-02T12:00:00Z")
print(sudden["finding"], sudden["strength"])
print(steady["finding"])
print(sudden["note"])

{'type': 'movement', 'level': 'sudden', 'people': 12} review
{'type': 'movement', 'level': 'steady', 'people': 10}
The level compares the busiest frame with the quietest. A sudden jump is a review, not an order.


## Plate clip

No plate reader is loaded, so a file never invents a registration. The demo complaint `DL01CV0001` exists only in the sample story, and the screen says it is not a real registration.

In [6]:
plate = build_event("plate", [{"car": 4}], "2026-11-02T12:00:00Z")
print(plate["finding"])
print(plate["note"])

{'type': 'plate', 'text': '', 'listed': False, 'complaint': ''}
No plate reader is loaded. A complaint is raised only when a reader returns a plate that is on the list.
